# Proyecto Big Data - Steam Reviews (Polars)


In [1]:
# ==========================================================
# CONFIGURACION DE RECOLECCION DE TIEMPOS
# ==========================================================
# Arquitectura A: 1 Master + 2 Workers (n2-standard-4)

tiempos_resultados = {}
arquitectura = "2_workers"


In [2]:
!gsutil cp gs://bigdata-2026-02/proyecto01/steam_reviews_500k.csv /tmp/steam_reviews_500k.csv

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://bigdata-2026-02/proyecto01/steam_reviews_500k.csv...
- [1 files][667.7 MiB/667.7 MiB]                                                
Operation completed over 1 objects/667.7 MiB.                                    


In [3]:
!pip install polars

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.8/865.8 kB 917.3 kB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 38.3 MB/s  0:00:016m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [polars]2m1/2 [polars]


In [4]:
import polars as pl

dfp = pl.read_csv("/tmp/steam_reviews_500k.csv")

print(dfp.shape)
print(dfp.schema)

(500000, 24)
Schema([('recommendationid', Int64), ('appid', Int64), ('game', String), ('author_steamid', Int64), ('author_num_games_owned', Int64), ('author_num_reviews', Int64), ('author_playtime_forever', Int64), ('author_playtime_last_two_weeks', Int64), ('author_playtime_at_review', Int64), ('author_last_played', Int64), ('language', String), ('review', String), ('timestamp_created', Int64), ('timestamp_updated', Int64), ('voted_up', Int64), ('votes_up', Int64), ('votes_funny', Int64), ('weighted_vote_score', Float64), ('comment_count', Int64), ('steam_purchase', Int64), ('received_for_free', Int64), ('written_during_early_access', Int64), ('hidden_in_steam_china', Int64), ('steam_china_location', String)])


## Consulta 1 - Exploración y validación del dataset

Objetivo: conocer dimensiones, estructura, tipos de datos y valores nulos del dataset.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [5]:
# ==========================================================
# CONSULTA 1 - POLARS
# ==========================================================

import time

inicio = time.time()

print("===== POLARS =====")


print("\nDimensiones del dataset")

print("Filas:", dfp.shape[0])
print("Columnas:", dfp.shape[1])


print("\nEstructura de datos")

print(dfp.schema)


print("\nValores nulos")

print(dfp.null_count())


fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Polars_Consulta_1"] = fin - inicio


===== POLARS =====

Dimensiones del dataset
Filas: 500000
Columnas: 24

Estructura de datos
Schema([('recommendationid', Int64), ('appid', Int64), ('game', String), ('author_steamid', Int64), ('author_num_games_owned', Int64), ('author_num_reviews', Int64), ('author_playtime_forever', Int64), ('author_playtime_last_two_weeks', Int64), ('author_playtime_at_review', Int64), ('author_last_played', Int64), ('language', String), ('review', String), ('timestamp_created', Int64), ('timestamp_updated', Int64), ('voted_up', Int64), ('votes_up', Int64), ('votes_funny', Int64), ('weighted_vote_score', Float64), ('comment_count', Int64), ('steam_purchase', Int64), ('received_for_free', Int64), ('written_during_early_access', Int64), ('hidden_in_steam_china', Int64), ('steam_china_location', String)])

Valores nulos
shape: (1, 24)
┌─────────────┬───────┬──────┬─────────────┬───┬────────────┬────────────┬────────────┬────────────┐
│ recommendat ┆ appid ┆ game ┆ author_stea ┆ … ┆ received_f ┆ written

## Consulta 2 - Eliminación de duplicados

Objetivo: eliminar registros repetidos considerando author_steamid, appid y review.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [6]:
# ==========================================================
# CONSULTA 2 - ELIMINACIÓN DE DUPLICADOS CON POLARS
# ==========================================================

import time

inicio = time.time()

print("===== POLARS =====")


# Cantidad inicial de registros

registros_iniciales = dfp.shape[0]

print("Registros iniciales:",
      registros_iniciales)


# Eliminación de duplicados

dfp_clean = dfp.unique(
    subset=[
        "author_steamid",
        "appid",
        "review"
    ]
)


# Cantidad final

registros_finales = dfp_clean.shape[0]


print("Registros después de eliminar duplicados:",
      registros_finales)


print("Duplicados eliminados:",
      registros_iniciales - registros_finales)


fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")

tiempos_resultados["Polars_Consulta_2"] = fin - inicio


===== POLARS =====
Registros iniciales: 500000
Registros después de eliminar duplicados: 315809
Duplicados eliminados: 184191

Tiempo de ejecución: 0.3209099769592285 segundos


## Consulta 3 - Tratamiento de valores nulos

Objetivo: identificar valores faltantes y limpiar registros sin información en review.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [7]:
# ==========================================================
# CONSULTA 3 - TRATAMIENTO DE VALORES NULOS CON POLARS
# ==========================================================

import time

inicio = time.time()

print("===== POLARS =====")


# Registros iniciales

registros_iniciales = dfp_clean.shape[0]

print("Registros iniciales:",
      registros_iniciales)


# Valores nulos antes

print("\nValores nulos antes:")

print(
    dfp_clean.null_count()
)


# Eliminación de registros sin review

dfp_null_clean = dfp_clean.drop_nulls(
    subset=["review"]
)


# Registros finales

registros_finales = dfp_null_clean.shape[0]


print("\nRegistros después de limpiar:",
      registros_finales)


print("Registros eliminados:",
      registros_iniciales - registros_finales)


# Valores nulos después

print("\nValores nulos después:")

print(
    dfp_null_clean.null_count()
)


fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Polars_Consulta_3"] = fin - inicio


===== POLARS =====
Registros iniciales: 315809

Valores nulos antes:
shape: (1, 24)
┌─────────────┬───────┬──────┬─────────────┬───┬────────────┬────────────┬────────────┬────────────┐
│ recommendat ┆ appid ┆ game ┆ author_stea ┆ … ┆ received_f ┆ written_du ┆ hidden_in_ ┆ steam_chin │
│ ionid       ┆ ---   ┆ ---  ┆ mid         ┆   ┆ or_free    ┆ ring_early ┆ steam_chin ┆ a_location │
│ ---         ┆ u32   ┆ u32  ┆ ---         ┆   ┆ ---        ┆ _access    ┆ a          ┆ ---        │
│ u32         ┆       ┆      ┆ u32         ┆   ┆ u32        ┆ ---        ┆ ---        ┆ u32        │
│             ┆       ┆      ┆             ┆   ┆            ┆ u32        ┆ u32        ┆            │
╞═════════════╪═══════╪══════╪═════════════╪═══╪════════════╪════════════╪════════════╪════════════╡
│ 0           ┆ 0     ┆ 22   ┆ 0           ┆ … ┆ 0          ┆ 0          ┆ 0          ┆ 315797     │
└─────────────┴───────┴──────┴─────────────┴───┴────────────┴────────────┴────────────┴────────────┘

Regist

## Consulta 4 - Transformación de variables

Objetivo: crear review_length como cantidad de caracteres de cada reseña.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [8]:
# ==========================================================
# CONSULTA 4 - TRANSFORMACIÓN DE VARIABLES CON POLARS
# ==========================================================

import time
import polars as pl

inicio = time.time()

print("===== POLARS =====")


# Crear variable review_length

dfp_transform = dfp_null_clean.with_columns(
    pl.col("review")
    .str.len_chars()
    .alias("review_length")
)


print("Registros procesados:",
      dfp_transform.shape[0])


print("\nEjemplo de transformación:")

print(
    dfp_transform.select(
        [
            "review",
            "review_length"
        ]
    ).head()
)


fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Polars_Consulta_4"] = fin - inicio


===== POLARS =====
Registros procesados: 315809

Ejemplo de transformación:
shape: (5, 2)
┌─────────────────────────────────┬───────────────┐
│ review                          ┆ review_length │
│ ---                             ┆ ---           │
│ str                             ┆ u32           │
╞═════════════════════════════════╪═══════════════╡
│ Знаете что?  Поиграв в Alan Wa… ┆ 425           │
│ let me romance Nick Valentine … ┆ 49            │
│ Как говорил мой дед: "Одна оши… ┆ 48            │
│  At 72 years of age, I've foun… ┆ 225           │
│ Assassin's Creed Origins  ❤ Au… ┆ 1104          │
└─────────────────────────────────┴───────────────┘

Tiempo de ejecución: 0.04973936080932617 segundos


## Consulta 5 - Filtrado de reseñas recomendadas

Objetivo: seleccionar registros donde voted_up sea igual a 1.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [9]:
# ==========================================================
# CONSULTA 5 - FILTRADO DE RESEÑAS RECOMENDADAS CON POLARS
# ==========================================================

import time
import polars as pl

inicio = time.time()

print("===== POLARS =====")


registros_iniciales = dfp_transform.shape[0]


print("Registros iniciales:",
      registros_iniciales)


# Filtrar reseñas recomendadas

dfp_positive = dfp_transform.filter(
    pl.col("voted_up") == 1
)


registros_finales = dfp_positive.shape[0]


print("Registros recomendados:",
      registros_finales)


print("Porcentaje de recomendaciones:",
      (registros_finales / registros_iniciales) * 100,
      "%")


print("\nEjemplo de datos:")

print(
    dfp_positive.select(
        [
            "game",
            "review",
            "voted_up"
        ]
    ).head()
)


fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Polars_Consulta_5"] = fin - inicio


===== POLARS =====
Registros iniciales: 315809
Registros recomendados: 266640
Porcentaje de recomendaciones: 84.43077936347603 %

Ejemplo de datos:
shape: (5, 3)
┌──────────────────────────┬─────────────────────────────────┬──────────┐
│ game                     ┆ review                          ┆ voted_up │
│ ---                      ┆ ---                             ┆ ---      │
│ str                      ┆ str                             ┆ i64      │
╞══════════════════════════╪═════════════════════════════════╪══════════╡
│ Alan Wake                ┆ Знаете что?  Поиграв в Alan Wa… ┆ 1        │
│ Fallout 4                ┆ let me romance Nick Valentine … ┆ 1        │
│ Project Zomboid          ┆ Как говорил мой дед: "Одна оши… ┆ 1        │
│ Hyper Dash               ┆  At 72 years of age, I've foun… ┆ 1        │
│ Assassin's Creed Origins ┆ Assassin's Creed Origins  ❤ Au… ┆ 1        │
└──────────────────────────┴─────────────────────────────────┴──────────┘

Tiempo de ejecución: 0.

## Consulta 6 - Cantidad de reseñas por videojuego

Objetivo: agrupar por game y calcular la cantidad total de reseñas.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [10]:
# ==========================================================
# CONSULTA 6 - CANTIDAD DE RESEÑAS POR VIDEOJUEGO - POLARS
# ==========================================================

import time
import polars as pl

inicio = time.time()

print("===== POLARS =====")


dfp_game_reviews = (
    dfp_transform
    .group_by("game")
    .agg(
        pl.len()
        .alias("total_reviews")
    )
    .sort(
        "total_reviews",
        descending=True
    )
)


print("Cantidad de videojuegos procesados:",
      dfp_game_reviews.shape[0])


print("\nTop videojuegos por cantidad de reseñas:")

print(
    dfp_game_reviews.head(10)
)


fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Polars_Consulta_6"] = fin - inicio


===== POLARS =====
Cantidad de videojuegos procesados: 22992

Top videojuegos por cantidad de reseñas:
shape: (10, 2)
┌────────────────────────────────┬───────────────┐
│ game                           ┆ total_reviews │
│ ---                            ┆ ---           │
│ str                            ┆ u32           │
╞════════════════════════════════╪═══════════════╡
│ Counter-Strike 2               ┆ 1923          │
│ PUBG: BATTLEGROUNDS            ┆ 1513          │
│ Stardew Valley                 ┆ 1413          │
│ Terraria                       ┆ 1264          │
│ The Witcher 3: Wild Hunt       ┆ 1228          │
│ Grand Theft Auto V             ┆ 1228          │
│ Tom Clancy's Rainbow Six Siege ┆ 1224          │
│ The Forest                     ┆ 1168          │
│ Wallpaper Engine               ┆ 1093          │
│ Rust                           ┆ 1034          │
└────────────────────────────────┴───────────────┘

Tiempo de ejecución: 0.045716285705566406 segundos


## Consulta 7 - Porcentaje de recomendación por videojuego

Objetivo: calcular porcentaje de reseñas positivas por juego usando voted_up.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [11]:
# ==========================================================
# CONSULTA 7 - PORCENTAJE DE RECOMENDACIÓN POR VIDEOJUEGO
# POLARS
# ==========================================================

import time
import polars as pl


inicio = time.time()

print("===== POLARS =====")


dfp_recommendation = (
    dfp_transform
    .group_by("game")
    .agg(
        [
            pl.len().alias("total_reviews"),

            (
                pl.col("voted_up")
                .sum()
            )
            .alias("positive_reviews")
        ]
    )
    .with_columns(
        (
            pl.col("positive_reviews") /
            pl.col("total_reviews") * 100
        )
        .alias("recommendation_percentage")
    )
    .sort(
        "recommendation_percentage",
        descending=True
    )
)


print("Videojuegos procesados:",
      dfp_recommendation.shape[0])


print("\nTop juegos recomendados:")

print(
    dfp_recommendation.head(10)
)


fin = time.time()

print("\nTiempo de ejecución:",
      fin-inicio,
      "segundos")
tiempos_resultados["Polars_Consulta_7"] = fin - inicio


===== POLARS =====
Videojuegos procesados: 22992

Top juegos recomendados:
shape: (10, 4)
┌─────────────────────────────────┬───────────────┬──────────────────┬───────────────────────────┐
│ game                            ┆ total_reviews ┆ positive_reviews ┆ recommendation_percentage │
│ ---                             ┆ ---           ┆ ---              ┆ ---                       │
│ str                             ┆ u32           ┆ i64              ┆ f64                       │
╞═════════════════════════════════╪═══════════════╪══════════════════╪═══════════════════════════╡
│ Football Coach: College Dynast… ┆ 1             ┆ 1                ┆ 100.0                     │
│ CUCKOLD SIMULATOR: Covid-19 Ma… ┆ 1             ┆ 1                ┆ 100.0                     │
│ Wire Lips                       ┆ 1             ┆ 1                ┆ 100.0                     │
│ Choice of the Deathless Demo    ┆ 1             ┆ 1                ┆ 100.0                     │
│ Impossible Creatu

## Consulta 8 - Promedio de horas jugadas por videojuego

Objetivo: calcular promedio de author_playtime_forever por juego.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [12]:
# ==========================================================
# CONSULTA 8 - PROMEDIO DE HORAS JUGADAS POR VIDEOJUEGO
# POLARS
# ==========================================================

import time
import polars as pl


inicio = time.time()

print("===== POLARS =====")


dfp_playtime = (
    dfp_transform
    .group_by("game")
    .agg(
        pl.col("author_playtime_forever")
        .mean()
        .alias("avg_playtime_minutes")
    )
    .sort(
        "avg_playtime_minutes",
        descending=True
    )
)


print("Videojuegos procesados:",
      dfp_playtime.shape[0])


print("\nVideojuegos con mayor promedio de juego:")

print(
    dfp_playtime.head(10)
)


fin = time.time()

print("\nTiempo de ejecución:",
      fin-inicio,
      "segundos")
tiempos_resultados["Polars_Consulta_8"] = fin - inicio


===== POLARS =====
Videojuegos procesados: 22992

Videojuegos con mayor promedio de juego:
shape: (10, 2)
┌─────────────────────────────────┬──────────────────────┐
│ game                            ┆ avg_playtime_minutes │
│ ---                             ┆ ---                  │
│ str                             ┆ f64                  │
╞═════════════════════════════════╪══════════════════════╡
│ Oops!!! I Slept With Your Mom   ┆ 2.181697e6           │
│ Legions of Ashworld             ┆ 1.982834e6           │
│ Aquarium Simulator              ┆ 1.687347e6           │
│ WalkinVR                        ┆ 1.366674e6           │
│ Houdini Indie                   ┆ 1264521.5            │
│ Oh, you touch my balls ( ͡° ͜ʖ…   ┆ 1.247696e6           │
│ The Putinland: Divide & Conque… ┆ 1.114523e6           │
│ MachineCraft                    ┆ 987286.0             │
│ Solitaire Forever II            ┆ 945538.0             │
│ Crusaders of the Lost Idols     ┆ 903523.0             │
└──────

## Consulta 9 - Longitud promedio de reseñas por videojuego

Objetivo: calcular promedio de review_length agrupado por game.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [13]:
# ==========================================================
# CONSULTA 9 - LONGITUD PROMEDIO DE RESEÑAS POR VIDEOJUEGO
# POLARS
# ==========================================================

import time
import polars as pl


inicio = time.time()

print("===== POLARS =====")


dfp_review_length = (
    dfp_transform
    .group_by("game")
    .agg(
        pl.col("review_length")
        .mean()
        .alias("avg_review_length")
    )
    .sort(
        "avg_review_length",
        descending=True
    )
)


print("Videojuegos procesados:",
      dfp_review_length.shape[0])


print("\nVideojuegos con reseñas más extensas:")

print(
    dfp_review_length.head(10)
)


fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Polars_Consulta_9"] = fin - inicio


===== POLARS =====
Videojuegos procesados: 22992

Videojuegos con reseñas más extensas:
shape: (10, 2)
┌────────────────────────┬───────────────────┐
│ game                   ┆ avg_review_length │
│ ---                    ┆ ---               │
│ str                    ┆ f64               │
╞════════════════════════╪═══════════════════╡
│ The Knight Witch       ┆ 8000.0            │
│ Mary Skelter 2         ┆ 8000.0            │
│ Wayward Strand         ┆ 8000.0            │
│ Indie Game Battle      ┆ 8000.0            │
│ Qbeh-1: The Atlas Cube ┆ 8000.0            │
│ Azusa Online           ┆ 7999.0            │
│ Deadly Flare           ┆ 7999.0            │
│ Alterium Shift         ┆ 7998.0            │
│ Dynopunk               ┆ 7996.0            │
│ The Companion          ┆ 7996.0            │
└────────────────────────┴───────────────────┘

Tiempo de ejecución: 0.03418564796447754 segundos


## Consulta 10 - Ranking de videojuegos

Objetivo: ordenar videojuegos considerando porcentaje de recomendación y cantidad de reseñas.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [14]:
# ==========================================================
# CONSULTA 10 - RANKING DE VIDEOJUEGOS
# POLARS
# ==========================================================

import time
import polars as pl


inicio = time.time()

print("===== POLARS =====")


dfp_ranking = (
    dfp_transform
    .group_by("game")
    .agg(
        [
            pl.len()
            .alias("total_reviews"),

            pl.col("voted_up")
            .sum()
            .alias("positive_reviews")
        ]
    )
    .with_columns(
        (
            pl.col("positive_reviews") /
            pl.col("total_reviews") * 100
        )
        .alias("recommendation_percentage")
    )
    .filter(
        pl.col("total_reviews") >= 100
    )
    .sort(
        [
            "recommendation_percentage",
            "total_reviews"
        ],
        descending=True
    )
)


print("Videojuegos rankeados:",
      dfp_ranking.shape[0])


print("\nTop videojuegos:")

print(
    dfp_ranking.head(10)
)


fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Polars_Consulta_10"] = fin - inicio


===== POLARS =====
Videojuegos rankeados: 609

Top videojuegos:
shape: (10, 4)
┌─────────────────────────────────┬───────────────┬──────────────────┬───────────────────────────┐
│ game                            ┆ total_reviews ┆ positive_reviews ┆ recommendation_percentage │
│ ---                             ┆ ---           ┆ ---              ┆ ---                       │
│ str                             ┆ u32           ┆ i64              ┆ f64                       │
╞═════════════════════════════════╪═══════════════╪══════════════════╪═══════════════════════════╡
│ Left 4 Dead 2                   ┆ 821           ┆ 821              ┆ 100.0                     │
│ Subnautica                      ┆ 807           ┆ 807              ┆ 100.0                     │
│ Mirror                          ┆ 604           ┆ 604              ┆ 100.0                     │
│ Plants vs. Zombies: Game of th… ┆ 532           ┆ 532              ┆ 100.0                     │
│ Mount & Blade: Warband      

In [15]:
# ==========================================================
# EXPORTACION AUTOMATIZADA DE RESULTADOS A GOOGLE CLOUD STORAGE
# ==========================================================

!pip install -q gcsfs fsspec

import pandas as pd

framework = "polars"

df_tiempos = pd.DataFrame(
    list(tiempos_resultados.items()),
    columns=["Consulta", "Tiempo_segundos"]
)

print(df_tiempos)

ruta_salida = f"gs://bigdata-2026-02/proyecto01/tiempos_{framework}_{arquitectura}.csv"

df_tiempos.to_csv(ruta_salida, index=False)

print(f"\nResultados exportados a: {ruta_salida}")


             Consulta  Tiempo_segundos
0   Polars_Consulta_1         0.010429
1   Polars_Consulta_2         0.320910
2   Polars_Consulta_3         0.003756
3   Polars_Consulta_4         0.049739
4   Polars_Consulta_5         0.008007
5   Polars_Consulta_6         0.045716
6   Polars_Consulta_7         0.029953
7   Polars_Consulta_8         0.032953
8   Polars_Consulta_9         0.034186
9  Polars_Consulta_10         0.031416

Resultados exportados a: gs://bigdata-2026-02/proyecto01/tiempos_polars_2_workers.csv
